In [1]:
# Initialize Otter
import otter
grader = otter.Notebook("hw7.ipynb")

# CPSC 330 - Applied Machine Learning 

## Homework 7: Word embeddings and topic modeling 

**Due date: See [deliverable due dates](https://ubc-cs.github.io/cpsc330-2025W2/#deliverable-due-dates-tentative)**.

## Imports

In [76]:
import os

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

import re

<br><br>

<!-- BEGIN QUESTION -->

<!-- BEGIN QUESTION -->

<div class="alert alert-info">
    
## Instructions
rubric={points}

You will earn points for following these instructions and successfully submitting your work on Gradescope.  

### Group wotk instructions

**You may work with a partner on this homework and submit your assignment as a group.** Below are some instructions on working as a group.  
- The maximum group size is 2.
  
- Use group work as an opportunity to collaborate and learn new things from each other. 
- Be respectful to each other and make sure you understand all the concepts in the assignment well. 
- It's your responsibility to make sure that the assignment is submitted by one of the group members before the deadline. 
- You can find the instructions on how to do group submission on Gradescope [here](https://help.gradescope.com/article/m5qz2xsnjy-student-add-group-members).
- If you would like to use late tokens for the homework, all group members must have the necessary late tokens available. Please note that the late tokens will be counted for all members of the group.   


### General submission instructions

- Please **read carefully
[Use of Generative AI policy](https://ubc-cs.github.io/cpsc330-2025W2/syllabus#use-of-generative-ai-in-the-course)** before starting the homework assignment. 
- **Run all cells before submitting:** Go to `Kernel -> Restart Kernel and Clear All Outputs`, then select `Run -> Run All Cells`. This ensures your notebook runs cleanly from start to finish without errors.
  
- **Submit your files on Gradescope.**  
   - Upload only your `.ipynb` file **with outputs displayed** and any required output files.
     
   - Do **not** submit other files from your repository.  
   - If you need help, see the [Gradescope Student Guide](https://lthub.ubc.ca/guides/gradescope-student-guide/).  
- **Check that outputs render properly.**  
   - Make sure all plots and outputs appear in your submission.
     
   - If your `.ipynb` file is too large and doesn't render on Gradescope, also upload a PDF or HTML version so the TAs can view your work.  
- **Keep execution order clean.**  
   - Execution numbers must start at "1" and increase in order.
     
   - Notebooks without visible outputs may not be graded.  
   - Out-of-order or missing execution numbers may result in mark deductions.  
- **Follow course submission guidelines:** Review the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions) for detailed guidance on completing and submitting assignments. 
   
</div>


_Points:_ 2

<!-- END QUESTION -->

<br><br><br><br>

## Exercise 1:  Exploring pre-trained word embeddings <a name="1"></a>
<hr>

In lecture 18, we talked about natural language processing (NLP). Using pre-trained word embeddings is very common in NLP. It has been shown that pre-trained word embeddings work well on a variety of text classification tasks. These embeddings are created by training a model like Word2Vec on a huge corpus of text such as a dump of Wikipedia or a dump of the web crawl. 

A number of pre-trained word embeddings are available out there. Some popular ones are: 

- [GloVe](https://nlp.stanford.edu/projects/glove/)
    * trained using [the GloVe algorithm](https://nlp.stanford.edu/pubs/glove.pdf) 
    * published by Stanford University 
- [fastText pre-trained embeddings for 294 languages](https://fasttext.cc/docs/en/pretrained-vectors.html) 
    * trained using the fastText algorithm
    * published by Facebook
    
In this exercise, you will be exploring GloVe Wikipedia pre-trained embeddings. The code below loads the word vectors trained on Wikipedia using an algorithm called Glove. You'll need `gensim` package in your cpsc330 conda environment to run the code below. 

```
> conda activate cpsc330
> conda install -c anaconda gensim
```

In [3]:
import gensim
import gensim.downloader

print(list(gensim.downloader.info()["models"].keys()))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


In [4]:
# This will take a while to run when you run it for the first time.
import gensim.downloader as api

glove_wiki_vectors = api.load("glove-wiki-gigaword-100")

[==================================================] 100.0% 128.1/128.1MB downloaded


In [6]:
len(glove_wiki_vectors)

400000

There are 400,000 word vectors in this pre-trained model. 

Now that we have GloVe Wiki vectors loaded in `glove_wiki_vectors`, let's explore the embeddings. 

<br><br>

<!-- BEGIN QUESTION -->

### 1.1 Word similarity using pre-trained embeddings
rubric={points}

**Your tasks:**

- Come up with a list of 4 words of your choice and find similar words to these words using `glove_wiki_vectors` embeddings.

<div class="alert alert-warning">

Solution_1.1
    
</div>

_Points:_ 2

In [9]:
words = ["king", "computer", "music", "city"]

for word in words:
    print(f"Top similar words to '{word}': {[word for (word, _similarity) in glove_wiki_vectors.most_similar(word, topn=5)]}")

Top similar words to 'king': ['prince', 'queen', 'son', 'brother', 'monarch']
Top similar words to 'computer': ['computers', 'software', 'technology', 'pc', 'hardware']
Top similar words to 'music': ['musical', 'songs', 'dance', 'pop', 'recording']
Top similar words to 'city': ['town', 'cities', 'where', 'area', 'downtown']


In [ ]:
...

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.2 Word similarity using pre-trained embeddings
rubric={points}

**Your tasks:**

1. Calculate cosine similarity for the following word pairs (`word_pairs`) using the [`similarity`](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=similarity#gensim.models.keyedvectors.KeyedVectors.similarity) method of `glove_wiki_vectors`.

In [10]:
word_pairs = [
    ("coast", "shore"),
    ("clothes", "closet"),
    ("old", "new"),
    ("smart", "intelligent"),
    ("dog", "cat"),
    ("tree", "lawyer"),
]

<div class="alert alert-warning">

Solution_1.2
    
</div>

_Points:_ 2

In [13]:
for w1, w2 in word_pairs:
    print(f"'{w1}' and '{w2}' similarity: {glove_wiki_vectors.similarity(w1, w2):.4f}")

'coast' and 'shore' similarity: 0.7000
'clothes' and 'closet' similarity: 0.5463
'old' and 'new' similarity: 0.6432
'smart' and 'intelligent' similarity: 0.7553
'dog' and 'cat' similarity: 0.8798
'tree' and 'lawyer' similarity: 0.0767


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.3 Representation of all words in English
rubric={points}

**Your tasks:**

1. The vocabulary size of Wikipedia embeddings is quite large. The `test_words` list below contains a few new words (called neologisms) and biomedical domain-specific abbreviations. Write code to check whether `glove_wiki_vectors` have representation for these words or not. 
> If a given word `word` is in the vocabulary, `word in glove_wiki_vectors` will return True. 

In [7]:
test_words = [
    "covididiot",
    "fomo",
    "frenemies",
    "anthropause",
    "photobomb",
    "selfie",
    "pxg",  # Abbreviation for pseudoexfoliative glaucoma
    "pacg",  # Abbreviation for primary angle closure glaucoma
    "cct",  # Abbreviation for central corneal thickness
    "escc",  # Abbreviation for esophageal squamous cell carcinoma
]

<div class="alert alert-warning">

Solution_1_3
    
</div>

_Points:_ 2

In [14]:
for word in test_words:
    print(f"{word} in wiki: {word in glove_wiki_vectors}")

covididiot in wiki: False
fomo in wiki: False
frenemies in wiki: True
anthropause in wiki: False
photobomb in wiki: False
selfie in wiki: False
pxg in wiki: False
pacg in wiki: False
cct in wiki: True
escc in wiki: True


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.4 Stereotypes and biases in embeddings
rubric={points}

Word vectors contain lots of useful information. But they also contain stereotypes and biases of the texts they were trained on. In the lecture, we saw an example of gender bias in Google News word embeddings. Here we are using pre-trained embeddings trained on Wikipedia data. 

**Your tasks:**

1. Explore whether there are any worrisome biases or stereotypes present in these embeddings by trying out at least 4 examples. You can use the following two methods or other methods of your choice to explore this. 
    - the `analogy` function below which gives word analogies (an example shown below)
    - [similarity](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=similarity#gensim.models.keyedvectors.KeyedVectors.similarity) or [distance](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=distance#gensim.models.keyedvectors.KeyedVectors.distances) methods (an example is shown below)

> Note that most of the recent embeddings are de-biased. But you might still observe some biases in them. Also, not all stereotypes present in pre-trained embeddings are necessarily bad. But you should be aware of them when you use them in your models. 

In [15]:
def analogy(word1, word2, word3, model=glove_wiki_vectors):
    """
    Returns analogy word using the given model.

    Parameters
    --------------
    word1 : (str)
        word1 in the analogy relation
    word2 : (str)
        word2 in the analogy relation
    word3 : (str)
        word3 in the analogy relation
    model :
        word embedding model

    Returns
    ---------------
        pd.dataframe
    """
    print("%s : %s :: %s : ?" % (word1, word2, word3))
    sim_words = model.most_similar(positive=[word3, word2], negative=[word1])
    return pd.DataFrame(sim_words, columns=["Analogy word", "Score"])

Examples of using analogy to explore biases and stereotypes.  

In [16]:
analogy("man", "doctor", "woman")

man : doctor :: woman : ?


,Analogy word,Score
0,nurse,0.773523
1,physician,0.718943
2,doctors,0.682433
3,patient,0.675068
4,dentist,0.672603
5,pregnant,0.664246
6,medical,0.652045
7,nursing,0.645348
8,mother,0.639333
9,hospital,0.638750


In [17]:
glove_wiki_vectors.similarity("aboriginal", "success")

np.float32(0.14283238)

In [18]:
glove_wiki_vectors.similarity("white", "success")

np.float32(0.35182396)

<div class="alert alert-warning">

Solution_1_4
    
</div>

_Points:_ 4

In [31]:
analogy("man", "leader", "woman")

man : leader :: woman : ?


,Analogy word,Score
0,opposition,0.688884
1,party,0.685079
2,lawmaker,0.651261
3,candidate,0.645046
4,democratic,0.622170
5,activist,0.600771
6,president,0.599504
7,leadership,0.595807
8,politician,0.595188
9,leaders,0.589962


In [46]:
analogy("man", "mathematics", "woman")

man : mathematics :: woman : ?


,Analogy word,Score
0,humanities,0.691818
1,sociology,0.688262
2,undergraduate,0.677993
3,biology,0.673875
4,physics,0.672595
5,sciences,0.664970
6,linguistics,0.655082
7,doctorate,0.647280
8,graduate,0.645335
9,phd,0.641917


In [64]:
analogy("man", "scream", "woman")

man : scream :: woman : ?


,Analogy word,Score
0,shriek,0.689712
1,screams,0.674535
2,cry,0.668148
3,yell,0.636522
4,shout,0.626864
5,holler,0.591468
6,cries,0.587510
7,shrieks,0.558155
8,screaming,0.557669
9,crying,0.547767


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.5 Discussion
rubric={points}

**Your tasks:**
1. Discuss your observations from 1.4. Are there any worrisome biases in these embeddings trained on Wikipedia?   
2. Give an example of how using embeddings with biases could cause harm in the real world.

<div class="alert alert-warning">

Solution_1_5
    
</div>

_Points:_ 4

The analogies tested in 1.4 range from subtle patterns, such as gendered framing of emotional expression (“scream” vs “shriek”), to more explicit stereotypes, such as associating men with STEM fields and women with humanities. These are worrisome because they reflect and reinforce existing societal biases present in the training data.

Using such biased embeddings in real-world applications can cause harm by at the very least potentially perpetuating these stereotypes and biases, and at worst perhaps being used in models that make key decisions such as those used for scanning resumes and making hiring decisions, where perhaps applications with that happen to have more technical terms (that are say associated more with masculine gender identities) than applications highlighting a mix of technical skills and soft skills (which may be associated within the model with more feminine gender identities) may be unfairly preferred by the model.

<!-- END QUESTION -->

<br><br><br><br>

## Exercise 2: Topic modeling 

The goal of topic modeling is discovering high-level themes in a large collection of texts. 

In this homework, you will explore topics in [the 20 newsgroups text dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_20newsgroups.html) using `scikit-learn`'s `LatentDirichletAllocation` (LDA) model. 

Usually, topic modeling is used for discovering abstract "topics" that occur in a collection of documents when you do not know the actual topics present in the documents. But 20 newsgroups text dataset is labeled with categories (e.g., sports, hardware, religion), and you will be able to cross-check the topics discovered by your model with these available topics. 

The starter code below loads the train and test portion of the data and convert the train portion into a pandas DataFrame. For speed, we will only consider documents with the following 8 categories. 

In [65]:
from sklearn.datasets import fetch_20newsgroups

In [66]:
cats = [
    "rec.sport.hockey",
    "rec.sport.baseball",
    "soc.religion.christian",
    "alt.atheism",
    "comp.graphics",
    "comp.windows.x",
    "talk.politics.mideast",
    "talk.politics.guns",
]  # We'll only consider these categories out of 20 categories for speed.

newsgroups_train = fetch_20newsgroups(
    subset="train", remove=("headers", "footers", "quotes"), categories=cats
)
X_news_train, y_news_train = newsgroups_train.data, newsgroups_train.target
df = pd.DataFrame(X_news_train, columns=["text"])
df["target"] = y_news_train
df["target_name"] = [
    newsgroups_train.target_names[target] for target in newsgroups_train.target
]
df

,text,target,target_name
0,"You know, I was reading 18 U.S.C. 922 and some...",6,talk.politics.guns
1,\n\n\nIt's not a bad question: I don't have an...,1,comp.graphics
2,"\nActuallay I don't, but on the other hand I d...",1,comp.graphics
3,"The following problem is really bugging me,\na...",2,comp.windows.x
4,\n\n This is the latest from UPI \n\n For...,7,talk.politics.mideast
...,...,...,...
4558,Hi Everyone ::\n\nI am looking for some soft...,1,comp.graphics
4559,Archive-name: x-faq/part3\nLast-modified: 1993...,2,comp.windows.x
4560,"\nThat's nice, but it doesn't answer the quest...",6,talk.politics.guns
4561,"Hi,\n I just got myself a Gateway 4DX-33V ...",2,comp.windows.x


In [67]:
newsgroups_train.target_names

['alt.atheism',
 'comp.graphics',
 'comp.windows.x',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast']

<br><br>

<!-- BEGIN QUESTION -->

### 2.1 Preprocessing using [spaCy](https://spacy.io/)
rubric={points}

Preprocessing is a crucial step before carrying out topic modeling and it markedly affects topic modeling results. In this exercise, you'll prepare the data using [spaCy](https://spacy.io/) for topic modeling. 

**Your tasks:** 

- Write code using [spaCy](https://spacy.io/) to preprocess the `text` column in the given dataframe `df` and save the processed text in a new column called `text_pp` within the same dataframe.

If you do not have spaCy in your course environment, you'll have to [install it](https://spacy.io/usage) and download the pretrained model en_core_web_md. 

`python -m spacy download en_core_web_md`


Note that there is no such thing as "perfect" preprocessing. You'll have to make your own judgments and decisions on which tokens are likely to be more informative for the given task. Some common text preprocessing steps for topic modeling include: 
- getting rid of slashes, new-line characters, or any other non-informative characters
- sentence segmentation and tokenization      
- replacing urls, email addresses, or numbers with generic tokens such as "URL",  "EMAIL", "NUM". 
- getting rid of other fairly unique tokens which are not going to help us in topic modeling  
- excluding stopwords and punctuation 
- lemmatization


> Check out [these available attributes](https://spacy.io/api/token#attributes) for `token` in spaCy which might help you with preprocessing. 

> You can also get rid of words with specific POS tags. [Here](https://universaldependencies.org/u/pos/) is the list of part-of-speech tags used in spaCy. 

> You may have to use regex to clean text before passing it to spaCy. Also, you might have to go back and forth between preprocessing in this exercise and and topic modeling in Exercise 2 before finalizing preprocessing steps. 

> Note that preprocessing the corpus might take some time. So here are a couple of suggestions: 1) During the debugging phase, work on a smaller subset of the data. 2) Once you finalize the preprocessing part, you might want to save the preprocessed data in a CSV and work with this CSV so that you don't run the preprocessing part every time you run the notebook. 
 


In [70]:
import spacy
nlp = spacy.load("en_core_web_md", disable=["parser", "ner"])

<div class="alert alert-warning">

Solution_2_1
    
</div>

_Points:_ 8

In [87]:
def preprocess(doc):
    # Used ChatGPT here to help look for regex generally used to identify emails and urls.
    doc = re.sub(r"\S+@\S+", "EMAIL", doc)
    doc = re.sub(r"http\S+|www\S+", "URL", doc)
    doc = re.sub(r"\d+", "NUM", doc)

    spacy_doc = nlp(doc)
    custom_stopwords = set([
        "num", "pnum", "people", "say", "know", "think", "time",
        "email", "send", "use", "thank", "come", "tell"
    ])

    tokens = []
    for token in spacy_doc:
        if (
            not token.is_stop
            and not token.is_punct
            and token.pos_ not in ["ADV", "PRON", "CCONJ", "PUNCT", "PART", "DET", "ADP", "SPACE"]
            and token.lemma_.lower() not in custom_stopwords
            and len(token) > 2
        ):
            tokens.append(token.lemma_.lower())

    return " ".join(tokens)

df["text_pp"] = [preprocess(text) for text in df["text"]]

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.2 Build a topic model using sklearn's LatentDirichletAllocation
rubric={points}

**Your tasks:**

1. Build LDA models on the preprocessed data using using [sklearn's `LatentDirichletAllocation`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html) and random state 42. Experiment with a few values for the number of topics (`n_components`). Pick a reasonable number for the number of topics and briefly justify your choice.

<div class="alert alert-warning">

Solution_2_2
    
</div>

_Points:_ 4

We already know that the dataset consists of 8 known categories and thus it makes the most sense to choose 8 as the reasonable number for topics. However, for the sake of experimentation, I tried n_components = 5 and 12 as well to see what we get if I go too low or too high. We can observe that for 5 components, the topics are a little mixed and unclear, e.g., with Topic 3: ['gun' 'file' 'entry' 'num' 'weapon'] mixing guns and computer related words. For 12 topics, we see more redundancy, e.g., topics 8 and 9 both being about religion, and 1 and 3 both about computers with unclear separation of content.

In [88]:
vec = CountVectorizer(max_features=2000, stop_words="english")
X = vec.fit_transform(df["text_pp"])

for k in [5, 8, 12]:
    lda = LatentDirichletAllocation(n_components=k, random_state=42)
    lda.fit(X)
    
    print(f"\nTop words for k={k}")
    for i, topic in enumerate(lda.components_):
        top_words = np.array(vec.get_feature_names_out())[np.argsort(topic)[::-1][:5]]
        print(f"Topic {i}: {top_words}")


Top words for k=5
Topic 0: ['game' 'team' 'year' 'play' 'good']
Topic 1: ['num' 'file' 'image' 'window' 'program']
Topic 2: ['armenian' 'israel' 'armenians' 'turkish' 'jews']
Topic 3: ['gun' 'file' 'entry' 'num' 'weapon']
Topic 4: ['god' 'believe' 'thing' 'jesus' 'good']

Top words for k=8
Topic 0: ['game' 'team' 'play' 'year' 'player']
Topic 1: ['file' 'image' 'window' 'program' 'server']
Topic 2: ['right' 'kill' 'want' 'start' 'thing']
Topic 3: ['gun' 'file' 'entry' 'law' 'firearm']
Topic 4: ['god' 'believe' 'jesus' 'thing' 'church']
Topic 5: ['post' 'look' 'good' 'work' 'mail']
Topic 6: ['israel' 'turkish' 'jews' 'armenian' 'israeli']
Topic 7: ['num' 'output' 'file' 'pnum' 'buf']

Top words for k=12
Topic 0: ['game' 'team' 'play' 'year' 'player']
Topic 1: ['image' 'file' 'window' 'program' 'server']
Topic 2: ['armenian' 'armenians' 'armenia' 'azerbaijan' 'university']
Topic 3: ['file' 'entry' 'section' 'program' 'rule']
Topic 4: ['woman' 'happen' 'want' 'leave' 'start']
Topic 5: ['

In [90]:
lda = LatentDirichletAllocation(n_components=8, random_state=42)
lda.fit(X)

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",8
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.3 Exploring word topic association
rubric={points}

**Your tasks:**
1. For the number of topics you picked in the previous exercise, show top 10 words for each of your topics and suggest labels for each of the topics (similar to how we came up with labels "health and nutrition", "fashion", and "machine learning" in the toy example we saw in class). 

> If your topics do not make much sense, you might have to go back to preprocessing in Exercise 2.1, improve it, and train your LDA model again. 

<div class="alert alert-warning">

Solution_2_3
    
</div>

_Points:_ 5

Based on the top 10 words displayed by the model, we can observe the topics:

Topic 0: Sports <br>
Topic 1: GUI <br>
Topic 2: Politics <br>
Topic 3: Guns <br>
Topic 4: Religion <br>
Topic 5: Communication <br>
Topic 6: Middle Eastern conflict <br>
Topic 7: Systems Programming <br>

In [91]:
feature_names = vec.get_feature_names_out()

for i, topic in enumerate(lda.components_):
    top_words = np.array(feature_names)[np.argsort(topic)[::-1][:10]]
    print(f"Topic {i}: {top_words}")

Topic 0: ['game' 'team' 'play' 'year' 'player' 'good' 'season' 'win' 'hockey'
 'league']
Topic 1: ['file' 'image' 'window' 'program' 'server' 'available' 'widget' 'num'
 'display' 'include']
Topic 2: ['right' 'kill' 'want' 'start' 'thing' 'happen' 'armenians' 'year'
 'government' 'leave']
Topic 3: ['gun' 'file' 'entry' 'law' 'firearm' 'weapon' 'crime' 'control' 'section'
 'state']
Topic 4: ['god' 'believe' 'jesus' 'thing' 'church' 'way' 'question' 'good' 'bible'
 'religion']
Topic 5: ['post' 'look' 'good' 'work' 'mail' 'book' 'need' 'like' 'read' 'write']
Topic 6: ['israel' 'turkish' 'jews' 'armenian' 'israeli' 'armenians' 'jewish' 'war'
 'state' 'government']
Topic 7: ['num' 'output' 'file' 'pnum' 'buf' 'entry' 'oname' 'printf' 'char'
 'stream']


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.4 Exploring document topic association
rubric={points}

**Your tasks:**
1. Show the document topic assignment of the first five documents from `df`.

<div class="alert alert-warning">

Solution_2_4
    
</div>

_Points:_ 5

In [95]:
doc_topics = lda.transform(X)

topics = ['sports', 'GUI', 'Politics', 'Guns', 'Religino', 'Communication', 'Middle East Conflict', 'Systems Programming']

for i in range(5):
    print(f"Doc {i}: {[f"{topic}: {score*100:.2f}%" for topic, score in zip(topics, doc_topics[i])]}")

Doc 0: ['sports: 0.36%', 'GUI: 0.36%', 'Politics: 0.36%', 'Guns: 84.12%', 'Religino: 0.36%', 'Communication: 13.73%', 'Middle East Conflict: 0.36%', 'Systems Programming: 0.36%']
Doc 1: ['sports: 0.25%', 'GUI: 0.25%', 'Politics: 40.92%', 'Guns: 0.25%', 'Religino: 0.25%', 'Communication: 57.61%', 'Middle East Conflict: 0.25%', 'Systems Programming: 0.25%']
Doc 2: ['sports: 0.26%', 'GUI: 9.01%', 'Politics: 0.26%', 'Guns: 0.26%', 'Religino: 0.26%', 'Communication: 89.43%', 'Middle East Conflict: 0.26%', 'Systems Programming: 0.26%']
Doc 3: ['sports: 0.78%', 'GUI: 71.20%', 'Politics: 24.10%', 'Guns: 0.78%', 'Religino: 0.78%', 'Communication: 0.78%', 'Middle East Conflict: 0.78%', 'Systems Programming: 0.78%']
Doc 4: ['sports: 0.50%', 'GUI: 0.50%', 'Politics: 0.50%', 'Guns: 0.50%', 'Religino: 0.50%', 'Communication: 0.50%', 'Middle East Conflict: 96.49%', 'Systems Programming: 0.50%']


<!-- END QUESTION -->

<br><br><br><br>

<!-- BEGIN QUESTION -->

## Exercise 3: Short answer questions 
<hr>

rubric={points}

1. Briefly explain how content-based filtering works in the context of recommender systems. 
2. Discuss at least two negative consequences of recommender systems.
3. What is transfer learning in natural language processing? Briefly explain.     

<div class="alert alert-warning">

Solution_3
    
</div>

_Points:_ 6

1. Content based filterting actually uses information we may have about given items in terms of specific features, and uses this to build models for each user based on their previous preferences to predict what features dictate whether or not they will like a new item with those specific features.


2. Main issue with content-based filtering is that users are only recommended what they already like, and a general issue with recommender systems is that popular items may be recommended too much with novel items not really being discovered as much.


3. In NLP specifically, transfer learning refers to things like training a model on a sufficiently large dataset so that it understands basic word structure, and then testing it on a more narrow application (such as legal use) including potentially fine-tuning with much lesser volume of downstream-task-specific data compared to the whole intial training dataset.

<!-- END QUESTION -->

<br><br><br><br>

Before submitting your assignment, please make sure you have followed all the instructions in the Submission Instructions section at the top. 

Here is a quick checklist before submitting: 

- [ ] Restart kernel, clear outputs, and run all cells from top to bottom.  
- [ ] `.ipynb` file runs without errors and contains all outputs.  
- [ ] Only `.ipynb` and required output files are uploaded (no extra files).  
- [ ] Execution numbers start at **1** and are in order.  
- [ ] If `.ipynb` is too large and doesn't render on Gradescope, also upload a PDF/HTML version.  
- [ ] Reviewed the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions).  

![](img/eva-well-done.png)